# Fuzzy Logic with scikit-fuzzy

Is 26°C hot? Most people would say "sort of". Ordinary code can't say "sort of": an `if temp > 25:` test is either `True` or `False`, so 24.9°C counts as not hot at all and 25.1°C counts as completely hot.

**Fuzzy logic** lets a value belong to a group *partly*. With the groups you'll build below, 26°C is a third warm and a third hot. That one idea makes it possible to write rules the way people talk ("if it's hot and humid, run the fan fast") and still get smooth, sensible answers. Fuzzy controllers run washing machines, air conditioners, camera autofocus and train brakes.

**scikit-fuzzy** (imported as `skfuzzy`) is Python's fuzzy logic toolkit. In this notebook you'll use it to build a controller for a room fan that decides its speed from the temperature and humidity. If arrays or charts are new to you, try [[A6AFAE39-B14C-4CD8-B9CB-23771BBF07C9|NumPy]] and [[F9D25507-BA02-4876-9F0C-CD35B389BC47|Matplotlib]] first.

# Setting Up

Fuzzy logic always works over a **universe**: every value a measurement could take. Our fan cares about room temperatures from 10°C to 40°C, so that range, as a NumPy array, is the temperature universe.

The parts of scikit-fuzzy you'll use, in the order this notebook meets them:

| Tool | What it's for |
|---|---|
| `fuzz.trimf`, `fuzz.trapmf`, `fuzz.gaussmf` | Drawing the shape of a fuzzy group |
| `fuzz.interp_membership` | How much one value belongs to a group |
| `fuzz.fuzzy_and`, `fuzz.fuzzy_or`, `fuzz.fuzzy_not` | Combining groups |
| `fuzz.defuzz` | Turning a fuzzy answer back into one number |
| `skfuzzy.control` | Building a complete controller from rules |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import skfuzzy as fuzz

temperature = np.linspace(10, 40, 301)   # 10.0, 10.1, ... 40.0 °C

print("Universe from", temperature[0], "to", temperature[-1], "°C")
print("Number of points:", len(temperature))

# Membership Functions

A **membership function** gives every value in the universe a score from 0 (not in the group) to 1 (completely in the group). scikit-fuzzy has a function for each common shape:

- `fuzz.trimf(universe, [a, b, c])` is a **triangle**: 0 at `a`, rising to 1 at `b`, back to 0 at `c`.
- `fuzz.trapmf(universe, [a, b, c, d])` is a **trapezoid**: it rises from `a` to `b`, stays at 1 until `c`, and falls to 0 at `d`. Repeating a corner (`[10, 10, ...]`) makes a flat edge.
- `fuzz.gaussmf(universe, mean, sigma)` is a smooth **bell curve**.

Notice how the groups overlap. Between 16°C and 20°C a room is a bit cold *and* a bit warm at the same time, just as people would describe it.

In [ ]:
cold = fuzz.trapmf(temperature, [10, 10, 15, 20])
warm = fuzz.trimf(temperature, [16, 22, 28])
hot = fuzz.trapmf(temperature, [24, 30, 40, 40])

plt.figure(figsize=(10, 4))
plt.plot(temperature, cold, color='steelblue', linewidth=2, label='Cold')
plt.plot(temperature, warm, color='seagreen', linewidth=2, label='Warm')
plt.plot(temperature, hot, color='crimson', linewidth=2, label='Hot')
plt.title('How Cold, Warm and Hot Is the Room?')
plt.xlabel('Temperature (°C)')
plt.ylabel('Membership')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

# Degrees of Membership

To ask how much one temperature belongs to a group, use `fuzz.interp_membership(universe, group, value)`. It reads the height of the group's curve at that value. This step is called **fuzzification**: turning an exact reading into fuzzy scores.

A single temperature can score in more than one group. The scores don't have to add up to 1, and a value can score 0 everywhere if no group covers it.

In [ ]:
for reading in [14, 18, 23, 26, 33]:
  c = fuzz.interp_membership(temperature, cold, reading)
  w = fuzz.interp_membership(temperature, warm, reading)
  h = fuzz.interp_membership(temperature, hot, reading)
  print(f"{reading}°C   cold {c:.2f}   warm {w:.2f}   hot {h:.2f}")

# Fuzzy AND, OR and NOT

Rules join conditions together, so fuzzy logic needs its own versions of `and`, `or` and `not`:

- **AND** takes the *smaller* score. If the room is 0.4 hot and 0.7 humid, it is 0.4 "hot and humid".
- **OR** takes the *larger* score.
- **NOT** is 1 minus the score.

`fuzz.fuzzy_or` and `fuzz.fuzzy_and` combine two whole groups and return a new universe and membership function. `fuzz.fuzzy_not` flips a group. Below, "warm or hot" covers everything from mild to scorching, and "not cold" is cold turned upside down.

In [ ]:
x, warm_or_hot = fuzz.fuzzy_or(temperature, warm, temperature, hot)
not_cold = fuzz.fuzzy_not(cold)

plt.figure(figsize=(10, 4))
plt.plot(x, warm_or_hot, color='darkorange', linewidth=3, label='Warm OR hot')
plt.plot(temperature, not_cold, color='purple', linestyle='--', linewidth=2, label='NOT cold')
plt.title('Combining Fuzzy Groups')
plt.xlabel('Temperature (°C)')
plt.ylabel('Membership')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

hot_score, humid_score = 0.4, 0.7
print("hot AND humid:", min(hot_score, humid_score))
print("hot OR humid:", max(hot_score, humid_score))
print("NOT hot:", 1 - hot_score)

# Defuzzification

A fan can't run at "0.6 medium and 0.3 fast". In the end it needs one speed. Turning a fuzzy result back into a single number is **defuzzification**.

When rules fire, each one cuts its output group off at the strength of the rule, and the pieces are joined into one shape. `fuzz.defuzz(universe, shape, mode)` then picks a number from that shape. The most popular mode, `'centroid'`, finds its balance point, like the spot where a cardboard cut-out of the shape would balance on a finger. `'mom'` (mean of maximum) picks the middle of the tallest part instead.

In [ ]:
speed = np.linspace(0, 100, 201)   # fan speed, %
medium = fuzz.trimf(speed, [20, 50, 80])
fast = fuzz.trapmf(speed, [60, 85, 100, 100])

# Suppose one rule says "medium" with strength 0.6, and another "fast" with 0.3.
shape = np.fmax(np.fmin(0.6, medium), np.fmin(0.3, fast))

centre = fuzz.defuzz(speed, shape, 'centroid')
peak = fuzz.defuzz(speed, shape, 'mom')
print(f"Centroid: {centre:.1f}%")
print(f"Mean of maximum: {peak:.1f}%")

plt.figure(figsize=(10, 4))
plt.fill_between(speed, shape, color='lightskyblue', label='Combined rule output')
plt.axvline(centre, color='crimson', linewidth=2, label=f'Centroid ({centre:.0f}%)')
plt.title('From a Fuzzy Shape to One Fan Speed')
plt.xlabel('Fan speed (%)')
plt.ylabel('Membership')
plt.legend()
plt.show()

# Building a Control System

Doing every step by hand gets long. The **`skfuzzy.control`** submodule does the whole job (fuzzify, apply the rules, combine and defuzzify) once you describe the problem:

- An **`Antecedent`** is an input, like the temperature. A **`Consequent`** is an output, like the fan speed.
- You give each one named groups with `variable['name'] = fuzz.trimf(...)`.
- A **`Rule`** links them: `ctrl.Rule(room_temp['hot'], fan_speed['fast'])`. Join conditions with `&` (and), `|` (or) and `~` (not).

Our fan has four rules, written almost the way a person would say them.

In [ ]:
from skfuzzy import control as ctrl

room_temp = ctrl.Antecedent(np.arange(10, 41, 1), 'temperature')
room_humidity = ctrl.Antecedent(np.arange(0, 101, 1), 'humidity')
fan_speed = ctrl.Consequent(np.arange(0, 101, 1), 'fan')

room_temp['cold'] = fuzz.trapmf(room_temp.universe, [10, 10, 15, 20])
room_temp['warm'] = fuzz.trimf(room_temp.universe, [16, 22, 28])
room_temp['hot'] = fuzz.trapmf(room_temp.universe, [24, 30, 40, 40])
room_humidity['dry'] = fuzz.trapmf(room_humidity.universe, [0, 0, 30, 60])
room_humidity['humid'] = fuzz.trapmf(room_humidity.universe, [40, 70, 100, 100])
fan_speed['slow'] = fuzz.trimf(fan_speed.universe, [0, 0, 40])
fan_speed['medium'] = fuzz.trimf(fan_speed.universe, [20, 50, 80])
fan_speed['fast'] = fuzz.trapmf(fan_speed.universe, [60, 85, 100, 100])

rules = [
  ctrl.Rule(room_temp['cold'], fan_speed['slow']),
  ctrl.Rule(room_temp['warm'] & room_humidity['dry'], fan_speed['slow']),
  ctrl.Rule(room_temp['warm'] & room_humidity['humid'], fan_speed['medium']),
  ctrl.Rule(room_temp['hot'], fan_speed['fast']),
]
# cache=False: scikit-fuzzy 0.5.0's result cache can give wrong answers over many runs
fan_sim = ctrl.ControlSystemSimulation(ctrl.ControlSystem(rules), cache=False)

room_temp.view()

# Running the Controller

A **`ControlSystemSimulation`** runs the rules. Set each input with `fan_sim.input['label'] = value`, call `fan_sim.compute()`, and read the answer from `fan_sim.output['label']`.

`fan_speed.view(sim=fan_sim)` shades the combined rule output and marks the speed the controller chose.

Drag the sliders to change the room. Try a warm room as it gets more and more humid, and notice the fan speeding up smoothly instead of jumping from one setting to the next.

In [ ]:
TEMPERATURE = 24 #@param {type:"slider", min:10, max:40, step:1}
HUMIDITY = 60 #@param {type:"slider", min:0, max:100, step:5}

fan_sim.input['temperature'] = TEMPERATURE
fan_sim.input['humidity'] = HUMIDITY
fan_sim.compute()

fan_speed.view(sim=fan_sim)

# The Whole Picture

Because the controller is just a function from (temperature, humidity) to fan speed, you can run it for every combination and draw the result as a colour map. This is called the controller's **control surface**. It runs the controller about 1,600 times, so give it a couple of seconds.

Where only one rule applies, such as any room below 16°C, the speed is flat. Where groups overlap, the speed blends gradually from one setting to the next. A version built from `if` statements would jump in a sharp step at every boundary instead, which is why fuzzy control suits fans, thermostats and car brakes.

In [ ]:
temps = np.arange(10, 40.5, 0.5)
humidities = np.arange(0, 101, 4)
speeds = np.zeros((len(humidities), len(temps)))

for row, h in enumerate(humidities):
  for col, t in enumerate(temps):
    fan_sim.input['temperature'] = t
    fan_sim.input['humidity'] = h
    fan_sim.compute()
    speeds[row, col] = fan_sim.output['fan']

plt.figure(figsize=(10, 5))
plt.imshow(speeds, origin='lower', aspect='auto', cmap='viridis', interpolation='bilinear',
           extent=[temps[0], temps[-1], humidities[0], humidities[-1]])
plt.colorbar(label='Fan speed (%)')
plt.title('Fan Speed for Every Room')
plt.xlabel('Temperature (°C)')
plt.ylabel('Humidity (%)')
plt.show()

# Check Your Understanding